In [12]:
from PIL import Image,ImageDraw, ImageFont

img = Image.new("RGB",(400,100),color=(255,255,255))
draw = ImageDraw.Draw(img)
font = ImageFont.truetype("/usr/share/fonts/noto/NotoSansDevanagari-Regular.ttf", 32)
draw.text((200, 50), "नमस्ते", font=font, fill=(0, 0, 0))
img.save("test.png")


In [13]:
from PIL import Image, ImageDraw, ImageFont
img = Image.new("RGB",(400,100),color=(255,255,255))
draw = ImageDraw.Draw(img)
font = ImageFont.truetype("usr/share/fonts/noto/NotoSansDevanagari-Regular.ttf",32)
bbox = draw.textbbox((0,0),"नमस्ते",font=font)
print(bbox)

(0, 0, 68, 30)


In [14]:
from PIL import Image, ImageDraw, ImageFont
img = Image.new("RGB",(400,100),color=(255,255,255))
draw = ImageDraw.Draw(img)
font = ImageFont.truetype("usr/share/fonts/noto/NotoSansDevanagari-Regular.ttf",32)
bbox = draw.textbbox((50,20),"नेपाली पाठ सिक्दै छु",font=font)
print(bbox)

(50, 20, 282, 58)


In [15]:
text = "नमस्ते"
x, y = 50, 20
bbox = draw.textbbox((0,0),text,font=font)
actual_bbox = (bbox[0]+x,bbox[1]+y,bbox[2]+x,bbox[3]+y)
draw.text((x,y),text,font=font,fill=(0,0,0))
draw.rectangle(actual_bbox,outline=(255,0,0),width=2)
img.save("test.png")

In [16]:
bbox = draw.textbbox((50, 20), text, font=font)
print(bbox)

(50, 20, 118, 50)


In [17]:
def render_image(text,font,font_size,pos,img_size):
    x,y=pos
    img = Image.new("RGB",img_size,color=(255,255,255))
    draw = ImageDraw.Draw(img)
    font = ImageFont.truetype(font,font_size)
    bbox = draw.textbbox((0,0),text,font=font)
    left,top,right,bottom = bbox
    w = right - left
    h = bottom - top 
    draw.text((x,y),text,font=font,fill=(0,0,0))
    return img,[x,y,w,h]

img, bbox = render_image(
    text="नमस्ते",
    font="/usr/share/fonts/noto/NotoSansDevanagari-Regular.ttf",
    font_size=32,
    pos=(50, 20),
    img_size=(400,100)
)
print(bbox)
img.save("test.png")

[50, 20, 68, 30]


In [18]:
import random

In [19]:
def render_image(text,font,font_size,img_size):
    img = Image.new("RGB",img_size,color=(255,255,255))
    draw = ImageDraw.Draw(img)
    font = ImageFont.truetype(font,font_size)
    bbox = draw.textbbox((0,0),text,font=font)
    left,top,right,bottom = bbox
    w = right - left
    h = bottom - top 
    max_x = img_size[0] - w
    max_y = img_size[1] - h
    x = random.randint(0, max_x)
    y = random.randint(0, max_y)
    draw.text((x,y),text,font=font,fill=(0,0,0))
    return img,[x,y,w,h]

img, bbox = render_image(
    text="नमस्ते",
    font="/usr/share/fonts/noto/NotoSansDevanagari-Regular.ttf",
    font_size=32,
    img_size=(400, 100)
)
print(bbox)
img.save("test.png")

[239, 26, 68, 30]


In [20]:
coc = {
    "info":{
        "year":2026
    },
    "images" : [
        {
            "id":1,
            "width":640,
            "height" : 480
        }
    ],
    "annotations": [],
    "categories":[
        {
            "id":1,
            "name":"devanagari_text"
        }
    ]
}

In [21]:
import os
import random
import json
from PIL import Image,ImageDraw,ImageFont

def render_image(text,font_path,font_size,img_size):
    img = Image.new("RGB",img_size,color=(255,255,255))
    draw = ImageDraw.Draw(img)

    font = ImageFont.truetype(font_path,font_size)
    bbox = draw.textbbox((0,0),text,font=font)
    left,top,right,bottom = bbox

    w = right - left
    h = bottom - top
    max_x = max(0, img_size[0]-w)
    max_y = max(0,img_size[1]-h)

    x = random.randint(0,max_x)
    y = random.randint(0,max_y)

    draw.text((x-left,y-top),text,font=font,fill=(0,0,0))
    img = add_noise(img, intensity=0.05)
    return img,[x,y,w,h]

# COCO skeleton
coco_dataset = {
    "info": {
        "year": 2026,
        "version": "1.0",
        "description": "Synthetic Devanagari Text Dataset",
        "date_created": "2026-06-04"
    },
    "images": [],
    "annotations": [],
    "categories": [
        {"id": 1, "name": "text", "supercategory": "character"}
    ]
}


In [22]:
font_file = "/usr/share/fonts/noto/NotoSansDevanagari-Regular.ttf"
output_dir = "dataset"
os.makedirs(output_dir, exist_ok=True)

for idx in range(1,6):
    file_name = f"devenagari_{idx}.png"
    img_path = os.path.join(output_dir,file_name)

    img, bbox = render_image(
    text="नमस्ते",
    font_path=font_file,
    font_size=32,
    img_size=(400, 100)
    )
    img.save(img_path)

    coco_dataset["images"].append({
        "id":idx,
        "width":400,
        "height":100,
        "file_name":file_name
    })

    bx,by,bw,bh = bbox
    polygon = [bx, by, bx + bw, by, bx + bw, by + bh, bx, by + bh]
    coco_dataset["annotations"].append({
        "id": 1000 + idx,
        "image_id": idx,
        "category_id": 1,
        "bbox": bbox,
        "area": float(bw * bh),
        "segmentation": [polygon],
        "iscrowd": 0
    })

# Save the final structured file
with open(os.path.join(output_dir, "annotations.json"), "w", encoding="utf-8") as f:
    json.dump(coco_dataset, f, indent=2, ensure_ascii=False)

print(f"Successfully generated 5 images and annotations.json in '{output_dir}/' folder.")



NameError: name 'add_noise' is not defined

In [ ]:
sentences = [
    "नमस्ते, मेरो नाम सौगत हो",
    "आज मौसम राम्रो छ",
    "नेपाल एक सुन्दर देश हो"
]

import random
text = random.choice(sentences)
print(text)

In [ ]:
import os
import random
import json
from PIL import Image,ImageDraw,ImageFont
def render_image(text,font_path,font_size,img_size):
    bg_mode = random.choice(["white", "colored", "gray"])
    # img = Image.new("RGB",img_size,color=(255,255,255))
    img = get_background(img_size[0], img_size[1], mode=bg_mode)
    draw = ImageDraw.Draw(img)

    font = ImageFont.truetype(font_path,font_size)
    bbox = draw.textbbox((0,0),text,font=font)
    left,top,right,bottom = bbox

    w = right - left
    h = bottom - top
    max_x = max(0, img_size[0]-w)
    max_y = max(0,img_size[1]-h)

    x = random.randint(0,max_x)
    y = random.randint(0,max_y)

    draw.text((x-left,y-top),text,font=font,fill=(0,0,0))
    img = add_noise(img,intensity=0.05)
    return img,[x,y,w,h]

# COCO skeleton
coco_dataset = {
    "info": {
        "year": 2026,
        "version": "1.0",
        "description": "Synthetic Devanagari Text Dataset",
        "date_created": "2026-06-04"
    },
    "images": [],
    "annotations": [],
    "categories": [
        {"id": 1, "name": "text", "supercategory": "character"}
    ]
}


In [ ]:
def get_background(width, height, mode="white"):
    if mode == "white":
        color = (255, 255, 255)
    elif mode == "colored":
        color = (255, 200, 150)  # light orange
    elif mode == "gray":
        color = (150, 150, 150)  # darker gray
    
    img = Image.new("RGB", (width, height), color=color)
    return img

In [ ]:
import numpy as np
def add_noise(img,intensity=0.05):
    arr = np.array(img,dtype=np.float32)
    noise = np.random.normal(0,intensity*255,arr.shape)
    arr = arr + noise
    arr = np.clip(arr,0,255).astype(np.uint8)
    return Image.fromarray(arr)
# Test
# bg = get_background(400, 100, "white")
# noisy_bg = add_noise(bg, intensity=0.1)
# noisy_bg.save("noisy_test.png")
    

In [ ]:
font_file = "/usr/share/fonts/noto/NotoSansDevanagari-Regular.ttf"
output_dir = "dataset"
os.makedirs(output_dir, exist_ok=True)

for idx in range(1,11):
    file_name = f"devenagari_{idx}.png"
    img_path = os.path.join(output_dir,file_name)
    font_size = random.choice(np.arange(10,41,4))
    img, bbox = render_image(
    text=random.choice(sentences),
    font_path=font_file,
    font_size=font_size,
    img_size=(400, 100)
    )
    img.save(img_path)

    coco_dataset["images"].append({
        "id":idx,
        "width":400,
        "height":100,
        "file_name":file_name
    })

    bx,by,bw,bh = bbox
    polygon = [bx, by, bx + bw, by, bx + bw, by + bh, bx, by + bh]
    coco_dataset["annotations"].append({
        "id": 1000 + idx,
        "image_id": idx,
        "category_id": 1,
        "bbox": bbox,
        "area": float(bw * bh),
        "segmentation": [polygon],
        "iscrowd": 0
    })

# Save the final structured file
with open(os.path.join(output_dir, "annotations.json"), "w", encoding="utf-8") as f:
    json.dump(coco_dataset, f, indent=2, ensure_ascii=False)

print(f"Successfully generated 11 images and annotations.json in '{output_dir}/' folder.")



In [24]:
from datasets import load_dataset

ds = load_dataset("himalaya-ai/cc100-nepali", split="train", streaming=True)
sentences = []
for i, row in enumerate(ds):
    for line in row["text"].split("\n"):
        line = line.strip()
        if 10 <= len(line) <= 100:
            sentences.append(line)
        if len(sentences) >= 500:
            break
    if len(sentences) >= 500:
        break

print(f"Loaded {len(sentences)} sentences")

Loaded 500 sentences


In [26]:
!for font in /usr/share/fonts/noto/NotoSansDevanagari-Regular.ttf \
  /usr/share/fonts/noto/NotoSerifDevanagari-Regular.ttf \
  /usr/share/fonts/ibm-plex-sans-devanagari/IBMPlexSansDevanagari-Regular.ttf \
  /usr/share/fonts/anek-devanagari/AnekDevanagari[wdth,wght].ttf \
  /usr/share/fonts/lohit-devanagari/Lohit-Devanagari.ttf \
  /usr/share/fonts/tiro-devanagari-hindi/TiroDevanagariHindi-Regular.ttf; do
  test -f "$font" && echo "OK: $font" || echo "MISSING: $font"
done

IndentationError: unexpected indent (4173079497.py, line 2)